In [ ]:
import os
import requests
from datetime import datetime

def get_previous_month(year, month, n):
    """
    Retorna o ano e mês subtraindo n meses da data informada.
    Se n==0, retorna o mês atual.
    """
    month -= n
    while month <= 0:
        month += 12
        year -= 1
    return year, month

# Base da URL e padrão do nome do arquivo
base_url = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/geracao_usina_2_ho/"
file_template = "GERACAO_USINA-2_{year}_{month:02d}.parquet"

# Diretório de destino
dest_dir = r"C:\Users\joao.barbosa\Codigos\Geracao por usina\Data\geracao_base_horaria"
os.makedirs(dest_dir, exist_ok=True)

# Obter data atual e definir quantos meses queremos baixar (ex: 2 meses)
hoje = datetime.today()
meses_para_baixar = 2

# Loop para baixar o mês atual (i=0) e o mês anterior (i=1)
for i in range(0, meses_para_baixar):
    ano, mes = get_previous_month(hoje.year, hoje.month, i)
    file_name = file_template.format(year=ano, month=mes)
    url = base_url + file_name
    print(f"Tentando baixar: {url}")
    
    # Realiza o download
    response = requests.get(url, verify=False)
    if response.status_code == 200:
        # Monta o caminho completo para salvar o arquivo
        file_path = os.path.join(dest_dir, file_name)
        with open(file_path, 'wb') as f:
            f.write(response.content)
        print(f"Arquivo {file_name} baixado com sucesso em {file_path}.")
    else:
        print(f"Falha ao baixar {file_name}. Status: {response.status_code}")


In [1]:
import os
import pandas as pd

# Defina o caminho do diretório
diretorio = r'C:\Users\joao.barbosa\Codigos\Geracao por usina\Data\geracao_base_horaria'
diretorio_cadastro = r"C:\Users\joao.barbosa\Codigos\despacho_termo\Data\cadastro\usina_empresa.xlsx"

# Lista para armazenar os DataFrames
dataframes = []

# Itera sobre os arquivos no diretório
for arquivo in os.listdir(diretorio):
    if arquivo.endswith('.parquet'):
        caminho_completo = os.path.join(diretorio, arquivo)
        df = pd.read_parquet(caminho_completo)
        dataframes.append(df)

# Concatena todos os DataFrames
df_unido = pd.concat(dataframes, ignore_index=True)


geracao_usina = df_unido


caminho_cadastro = os.path.join(diretorio, 'cadastro.xlsx')


cadastro = pd.read_excel(diretorio_cadastro)

geracao_usina_termo = geracao_usina.loc[geracao_usina['nom_tipousina']=='TÉRMICA']



In [2]:
# Verifica se as colunas esperadas estão presentes
print("Colunas do cadastro:", cadastro.columns.tolist())

# 2. Filtrar df_agrupado para manter apenas as linhas cujo 'ceg' esteja presente no cadastro
df_filtrado = geracao_usina_termo[geracao_usina_termo['ceg'].isin(cadastro['ceg'])].copy()

print(f"Número de registros em df_agrupado: {len(geracao_usina_termo)}")
print(f"Número de registros filtrados: {len(df_filtrado)}")

# 3. Mesclar df_filtrado com o cadastro para trazer a coluna 'empresa'
# Mantemos apenas as colunas 'empresa' e 'ceg' do cadastro
df_final = pd.merge(df_filtrado, cadastro[['empresa', 'ceg']], on='ceg', how='left')

Colunas do cadastro: ['ceg', 'empresa']
Número de registros em df_agrupado: 5593680
Número de registros filtrados: 250248


In [3]:
df_final

,din_instante,id_subsistema,nom_subsistema,id_estado,nom_estado,cod_modalidadeoperacao,nom_tipousina,nom_tipocombustivel,nom_usina,id_ons,ceg,val_geracao,empresa
0,2022-01-01 00:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,MC2 Nova Venécia 2,MAUTNV,UTE.GN.MA.030196-5.01,0E-8,Eneva
1,2022-01-01 00:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,Maranhão III,MAUTM3,UTE.GN.MA.030800-5.01,0E-8,Eneva
2,2022-01-01 00:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,Maranhão V,MAUTM5,UTE.GN.MA.030203-1.01,0E-8,Eneva
3,2022-01-01 00:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Carvão,Porto do Itaqui,MAUTPI,UTE.CM.MA.029700-3.01,0E-8,Eneva
4,2022-01-01 00:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,Parnaíba IV,MAUTP4,UTE.GN.MA.031193-6.01,0E-8,Eneva
...,...,...,...,...,...,...,...,...,...,...,...,...,...
250243,2025-04-08 23:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,Maranhão IV,MAUTM4,UTE.GN.MA.030202-3.01,0E-8,Eneva
250244,2025-04-08 23:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,Maranhão V,MAUTM5,UTE.GN.MA.030203-1.01,0E-8,Eneva
250245,2025-04-08 23:00:00,N,NORTE,MA,MARANHAO,TIPO I,TÉRMICA,Gás,MC2 Nova Venécia 2,MAUTNV,UTE.GN.MA.030196-5.01,0E-8,Eneva
250246,2025-04-08 23:00:00,NE,NORDESTE,CE,CEARA,TIPO I,TÉRMICA,Carvão,Porto do Pecém II,CEUTPD,UTE.CM.CE.030098-5.01,0E-8,Eneva


In [4]:
import pandas as pd

# Converter 'din_instante' para datetime
df_final['din_instante'] = pd.to_datetime(df_final['din_instante'])

# Converter 'val_geracao' para numérico, transformando erros em NaN
df_final['val_geracao'] = pd.to_numeric(df_final['val_geracao'], errors='coerce')

# Criar colunas separadas para o ano e o mês
df_final['ano'] = df_final['din_instante'].dt.year
df_final['mes'] = df_final['din_instante'].dt.month

# Agrupar por 'nom_usina', 'ceg', 'nom_tipocombustivel', 'ano' e 'mes' e calcular a média de 'val_geracao'
resultado = df_final.groupby(['nom_usina', 'ceg', 'nom_tipocombustivel', 'ano', 'mes'])['val_geracao'].mean().reset_index()



In [5]:
resultado.to_excel(r"P:\João Marcos\dados_alta_freq\geracao termo\geracao_termo_base_horario.xlsx")